# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SajidurCodes/flyrank-ml-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import duckdb
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\Projects AI ML\flyrank-ml-starter")
load_dotenv(PROJECT_ROOT / ".env", override=True)
HF_TOKEN = os.getenv("HF_TOKEN")

con = duckdb.connect()
con.execute("CREATE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
FACT_MARCH = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet"
FACT_APRIL = f"{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet"
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
DECISION_CUTOFF = "2026-03-31"

# --- Rebuild the exact same MODEL_DATA frame from w05 ---
MODEL_DATA = con.sql(f"""
    WITH march AS (
        SELECT
            client_hash_id, content_hash_id,
            SUM(gsc_clicks) AS clicks_march,
            SUM(gsc_impressions) AS impressions_march,
            AVG(gsc_avg_position) AS avg_position_march,
            SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS ctr_march
        FROM read_parquet('{FACT_MARCH}')
        GROUP BY client_hash_id, content_hash_id
    ),
    april AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_april
        FROM read_parquet('{FACT_APRIL}')
        GROUP BY client_hash_id, content_hash_id
    ),
    content AS (
        SELECT
            client_hash_id, content_hash_id,
            content_type, main_intent, word_count, char_count,
            DATE_DIFF('day', content_updated_date, DATE '{DECISION_CUTOFF}') AS days_stale
        FROM read_parquet('{DIM_CONTENT}')
        WHERE is_published IS TRUE AND is_deleted IS FALSE
          AND content_updated_date <= DATE '{DECISION_CUTOFF}'
    ),
    position_benchmark AS (
        SELECT ROUND(avg_position_march) AS position_bucket, AVG(ctr_march) AS expected_ctr
        FROM march GROUP BY ROUND(avg_position_march)
    )
    SELECT
        m.client_hash_id, m.content_hash_id, m.clicks_march, a.clicks_april,
        m.ctr_march, m.avg_position_march,
        GREATEST(pb.expected_ctr - m.ctr_march, 0) AS ctr_gap,
        c.content_type, c.main_intent, c.word_count, c.char_count, c.days_stale,
        CASE WHEN (a.clicks_april - m.clicks_march) / NULLIF(m.clicks_march, 0) < -0.15 THEN 1 ELSE 0 END AS declined
    FROM march m
    JOIN april a USING (client_hash_id, content_hash_id)
    JOIN content c USING (client_hash_id, content_hash_id)
    JOIN position_benchmark pb ON ROUND(m.avg_position_march) = pb.position_bucket
    WHERE m.clicks_march >= 5
""").df()

FEATURE_COLS_NUMERIC = ["ctr_march", "avg_position_march", "ctr_gap", "word_count", "char_count", "days_stale"]
FEATURE_COLS_CATEGORICAL = ["content_type", "main_intent"]
TARGET_COL = "declined"

print("Rows:", len(MODEL_DATA))


Rows: 3290


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — "What Predicts Health?" (Random Forest feature importance, ML Appendix)

**The claim:** a Random Forest, holdout-tested, ranks Average Position, Impressions, and Scroll Depth as the top predictors of a page's Health Score.

**Where the label comes from:** Health Score is not an independently observed outcome — it's a composite the paper itself defines as a weighted sum of impressions, position, CTR, and scroll depth. The paper actually discloses this tension directly, noting the target is partly constructed from some of the same inputs being tested.

**My methodology question:** if roughly 80 of the composite's 100 points come directly from three of the model's own top features (impressions, position, and CTR each get explicit point allocations), how much of the reported 43%+32%+15% importance split is the model rediscovering its own scoring formula rather than finding an independent predictive relationship? A holdout split protects against overfitting noise, but it doesn't protect against this — the model would score similarly well on any holdout if the relationship is close to deterministic. **The constructive ask:** it would strengthen the finding to also report feature importance for predicting Health Score using *only* the inputs that aren't part of the formula (content age, word count, days since update) — that version would show whether there's a genuine external predictive signal underneath, or whether the model's skill is fully explained by the composite's own arithmetic. This is the same category of question I had to ask my own w05 model, where I excluded `backlinks`/`competition`/`search_volume` specifically because I couldn't rule out them reflecting post-decision-point state — here the concern is the reverse direction, features reflecting the *label's own construction* rather than a future state, but the effect on interpretability is the same.

### Finding 2 — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, ML Appendix)

**The claim:** a logistic regression separates growing from declining pages using Content Age, Days Since Update, and Days Visible as the strongest signals, at 71% holdout accuracy.

**Where the label comes from:** Trend Direction is defined elsewhere in the paper as a 30-day-vs-previous-30-day impression change (Up: >10% growth, Down: >10% decline). That's a within-portfolio comparison of two adjacent time windows, not a genuinely forward-looking prediction from an earlier decision point to a later, unseen outcome.

**My methodology question:** the methodology section states the ML split is an 80/20 holdout but doesn't say whether it's row-random or grouped/time-aware. Two specific risks follow directly from that gap, both things I had to explicitly test and rule out in my own w06 Section 2:
1. **Client leakage** — if pages from the same brand appear in both the 80% and the 20%, the model could partly be learning brand-level patterns rather than page-level ones, since a brand's overall trajectory likely correlates across its own pages (I confirmed this exact risk in my own w05 model by comparing a naive random split against a client-grouped one).
2. **Temporal proximity between features and label** — "Days Visible" is called out as one of the strongest positive signals, and it's a metric measured over a similar recent window to the one the growth/decline label is itself computed from. It's worth asking whether "recent visibility" as a feature and "recent impression change" as the label are measuring closely related things, which would inflate the reported 71% without necessarily reflecting a signal usable *before* the outcome window closes.

**The constructive ask:** reporting the split method explicitly (grouped by brand, or not) and confirming the feature window ends strictly before the label window begins, would let a reader trust the 71% as a genuinely predictive number rather than a partially descriptive one — the same distinction the paper itself is careful to draw for the Random Forest health-score model just above it.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

w05 already used a client-grouped split from the start, so the honest "before" here is the naive random row-level split — the mistake this section exists to catch. "After" is the client-grouped split already used in w05. Showing both side by side proves why the grouped split mattered, not just that one was used.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

def precision_at_k(y_true, scores, k_frac=0.10):
    k = max(1, int(len(scores) * k_frac))
    top_k_idx = np.argsort(scores)[::-1][:k]
    return y_true.iloc[top_k_idx].mean()

def build_pipeline():
    preprocess = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), FEATURE_COLS_NUMERIC),
        ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="missing")), ("encode", OneHotEncoder(handle_unknown="ignore"))]), FEATURE_COLS_CATEGORICAL),
    ])
    return Pipeline([("prep", preprocess), ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])

X_cols = FEATURE_COLS_NUMERIC + FEATURE_COLS_CATEGORICAL

# --- BEFORE: naive random row-level split (ignores that pages share clients) ---
train_naive, test_naive = train_test_split(MODEL_DATA, test_size=0.25, random_state=42)
model_naive = build_pipeline()
model_naive.fit(train_naive[X_cols], train_naive[TARGET_COL])
scores_naive = model_naive.predict_proba(test_naive[X_cols])[:, 1]
naive_client_overlap = set(train_naive["client_hash_id"]) & set(test_naive["client_hash_id"])

# --- AFTER: client-grouped split (same as w05) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(MODEL_DATA, groups=MODEL_DATA["client_hash_id"]))
train_grouped, test_grouped = MODEL_DATA.iloc[train_idx], MODEL_DATA.iloc[test_idx]
model_grouped = build_pipeline()
model_grouped.fit(train_grouped[X_cols], train_grouped[TARGET_COL])
scores_grouped = model_grouped.predict_proba(test_grouped[X_cols])[:, 1]
grouped_client_overlap = set(train_grouped["client_hash_id"]) & set(test_grouped["client_hash_id"])

before_after = pd.DataFrame({
    "split_type": ["naive random (BEFORE)", "client-grouped (AFTER)"],
    "client_overlap_train_test": [len(naive_client_overlap), len(grouped_client_overlap)],
    "precision_at_10pct": [
        precision_at_k(test_naive[TARGET_COL].reset_index(drop=True), scores_naive),
        precision_at_k(test_grouped[TARGET_COL].reset_index(drop=True), scores_grouped),
    ],
    "roc_auc": [
        roc_auc_score(test_naive[TARGET_COL], scores_naive),
        roc_auc_score(test_grouped[TARGET_COL], scores_grouped),
    ],
})
print(before_after.to_string(index=False))



            split_type  client_overlap_train_test  precision_at_10pct  roc_auc
 naive random (BEFORE)                         17            0.695122 0.528881
client-grouped (AFTER)                          0            0.800000 0.530589


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Three checks against the exact feature set the model actually uses:
1. **Temporal check** — every feature's underlying date confirmed `<= 2026-03-31`; only the label is allowed to reach into April.
2. **Column-by-column justification** — one line per feature on why it was knowable at the March decision point.
3. **Target leakage check** — confirm `clicks_april` never appears among the model's feature columns.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



# --- 1. Temporal check ---
clamp_check = con.sql(f"""
    SELECT MAX(content_updated_date) AS latest_update_used
    FROM read_parquet('{DIM_CONTENT}')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
      AND content_updated_date <= DATE '{DECISION_CUTOFF}'
""").df()
print(clamp_check)
assert pd.Timestamp(clamp_check["latest_update_used"].iloc[0]) <= pd.Timestamp(DECISION_CUTOFF), "LEAKAGE: content_updated_date exceeds decision cutoff"
# --- 2. Column-by-column justification ---
justification = {
    "ctr_march":         "Aggregated GSC clicks/impressions logged during March itself.",
    "avg_position_march": "GSC position data logged daily during March.",
    "ctr_gap":            "Derived entirely from ctr_march and a March-only position benchmark.",
    "word_count":         "Static content property, treated as knowable at decision time. Caveat: dim_content is an unpartitioned snapshot with no per-field date, so if word count changed after March this value technically reflects the July export, not March. Disclosed residual risk, not a solved one.",
    "char_count":         "Same caveat as word_count.",
    "days_stale":         "Derived from content_updated_date, clamped to <= 2026-03-31 above. Verified leakage-free by the assert statement.",
}
for col, note in justification.items():
    print(f"{col}: {note}")

# --- 3. Target leakage check ---
assert "clicks_april" not in FEATURE_COLS_NUMERIC and "clicks_april" not in FEATURE_COLS_CATEGORICAL, \
    "LEAKAGE: the label's own input column is in the feature set"
print("\nConfirmed: clicks_april is used only to construct `declined`, never passed as a feature.")



  latest_update_used
0         2026-03-24
ctr_march: Aggregated GSC clicks/impressions logged during March itself.
avg_position_march: GSC position data logged daily during March.
ctr_gap: Derived entirely from ctr_march and a March-only position benchmark.
word_count: Static content property, treated as knowable at decision time. Caveat: dim_content is an unpartitioned snapshot with no per-field date, so if word count changed after March this value technically reflects the July export, not March. Disclosed residual risk, not a solved one.
char_count: Same caveat as word_count.
days_stale: Derived from content_updated_date, clamped to <= 2026-03-31 above. Verified leakage-free by the assert statement.

Confirmed: clicks_april is used only to construct `declined`, never passed as a feature.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim (from w05, Section 3):** something like *"Logistic Regression predicts which pages will decline."*

**Real w05 results table, client-grouped test set (n=159 rows):**

| method | precision@10% | roc_auc |
|---|---|---|
| w04 baseline rule | 0.667 | 0.579 |
| Logistic Regression | 0.800 | 0.530 |
| Random Forest | 0.667 | 0.489 |

**Rewritten:** "Within a client-held-out March-to-April test set of 159 pages, a logistic regression trained on CTR-gap-vs-position and content staleness identified the top decile of pages with a measured Precision@10% of 0.800, versus 0.667 for the Week-4 baseline rule and 0.667 for a Random Forest — an observed, directional improvement specifically at the top of the ranked queue, on this split and this label definition (>15% click drop, March to April). This is decision-support for prioritizing manual review, not a general or causal claim about which pages will decline."

**What the rewrite deliberately does NOT claim, and why that matters:**
- It does not say Logistic Regression is simply "the better model." On ROC-AUC — a measure of ranking quality across the *whole* test set, not just the top slice — the w04 baseline rule actually scored highest (0.579) and Logistic Regression scored lowest of the three (0.530). The honest picture is that Logistic Regression concentrates precision at the very top of the queue, while the baseline rule is more consistent across the full ranking. A bold claim would have hidden this trade-off.
- Random Forest is named as not earning its complexity: it tied the baseline on Precision@10% and scored worst on ROC-AUC. Per the w05 card's explicit warning against rewarding complexity alone, it should not be presented as an improvement.
- The test set is 159 rows. A precision figure built on roughly 16 top-decile pages can swing meaningfully on a handful of examples — the claim names this sample size rather than letting a reader assume a large-scale validated result.
- No claim about *why* a page declines — seasonality, algorithm changes, and competitor moves remain live alternative explanations, the same caveat flagged in the w03 data-limits section.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.